# KhadSan AI รอบ 3 : ผลเชื่อได้แค่ไหน

รอบ 2 ยังมีจุดอ่อน 3 อย่าง รอบนี้แก้ตรงนั้น โดย train/val/test ยังเป็น **ชุดเดิม** เทียบกับรอบก่อนได้ตรงๆ

| จุดอ่อนจากรอบ 2 | รอบ 3 ทำอะไร |
|---|---|
| ส้มเน่าแย่ลง (F1 0.908 → 0.859) | เพิ่มส้มเน่าจาก Roboflow `rotten-orange` 174 รูปเข้าชุดเทรน |
| เทรนแบบละครั้งเดียว (ล็อก seed 42) ไม่รู้ว่าคะแนนแกว่งแค่ไหน | เทรนค่าเดียวกัน 3 ครั้ง seed 42 / 2 / 3 (`10`, `11`, `12`) |
| ข้อสอบแหล่งอื่น (Kaggle) มาจากที่เดียวกับรูปที่เพิ่มเทรน และไม่มีมะนาว ทับทิม | ข้อสอบใหม่ 2 ชุด (ดูด้านล่าง) และให้โมเดลเก่า 5, 8 สอบด้วย |

ข้อสอบใหม่ (มีกรอบทุกลูก ไม่เคยใช้เทรน ตัดรูปที่คล้ายชุดเทรนออกแล้ว)
- `new_test` (ข้อสอบอิสระ) : ทับทิมสด 150 รูป จาก Roboflow `quality-detection`
- `same_source_test` (ดูประกอบ **ไม่ใช่ข้อสอบอิสระ**) : มะนาวสด/เน่า 298 รูป (`lime-quality`) + ทับทิมสด/เน่า 72 รูป (`pomegranate-aws5r`)
  2 ชุดนี้มาจากชุดถ่ายเดียวกับ Fruits Quality (เจอรูปเดียวกันเป๊ะ) เลยบอกได้แค่ว่าเก่งกับรูปแบบเดิมแค่ไหน

**ก่อนกด Run all**
1. push โฟลเดอร์ `train/` ล่าสุดขึ้น GitHub
2. Runtime > Change runtime type > **T4 GPU**
3. ใน Drive สร้างโฟลเดอร์ `KhadSan_detect_v3` แล้วอัปโหลด 3 ไฟล์ :
   - `KaggleBoxes_v3.zip` (รูป Kaggle 3,000 + พื้นหลัง 126 + ส้มเน่าใหม่ 174)
   - `NewTest.zip` (ข้อสอบใหม่ 2 ชุด)
   - `ExternalTest.zip` (ก๊อปจาก `KhadSan_detect_v2`)
4. dataset หลัก : ตั้ง Colab Secret `ROBOFLOW_API_KEY` **หรือ** อัปโหลด `FruitsQuality.zip` ไว้ในโฟลเดอร์เดียวกัน
5. ถ้ายังมีโฟลเดอร์ `KhadSan_detect_v2` ใน Drive โมเดล 5 จะถูกดึงมาสอบเทียบด้วย (ไม่มีก็ข้าม) ส่วนโมเดล 8 ใช้ไฟล์ใน `web/models/` ของ GitHub

ใช้เวลาราว 3-4 ชั่วโมง ถ้าหลุดกลางทาง กด Run all ใหม่ การทดลองที่เสร็จแล้วจะถูกข้าม

## 1) เช็กว่าได้ GPU
ถ้าขึ้นว่าไม่เจอ GPU : Runtime > Change runtime type > **T4 GPU** > Save แล้วกด Run all ใหม่

In [ ]:
import torch
if not torch.cuda.is_available():
    raise Exception("ไม่เจอ GPU ไปที่ Runtime > Change runtime type > เลือก T4 GPU แล้วกด Run all ใหม่")
print("ได้ GPU:", torch.cuda.get_device_name(0))

## 2) ติดตั้งไลบรารี
ล็อกรุ่น ultralytics ให้ตรงกับที่ทีมใช้ (8.4.191) ผลจะได้เทียบกันได้

In [ ]:
!pip install -q ultralytics==8.4.191 roboflow
!yolo settings sync=False
import ultralytics
print("ultralytics", ultralytics.__version__)
if ultralytics.__version__ != "8.4.191":
    raise Exception("รุ่น ultralytics ไม่ตรง ให้ Runtime > Restart session แล้วกด Run all ใหม่")

## 3) ดึงโค้ดเทรนจาก GitHub ของทีม
ต้อง **push โค้ดล่าสุดขึ้น GitHub ก่อน** ช่องนี้จะเช็กให้ว่าโค้ดที่ดึงมาเป็นรุ่นที่ใช้กับรอบนี้ได้

In [ ]:
import os
REPO = "https://github.com/srirat-intalai/khadsan-ai"

# ดึงครั้งแรก = clone / รันซ้ำ = อัปเดตเป็นโค้ดล่าสุด
if os.path.isdir("/content/khadsan-ai"):
    !git -C /content/khadsan-ai pull -q
else:
    !git clone -q {REPO} /content/khadsan-ai

CODE = "/content/khadsan-ai/train"
!git -C /content/khadsan-ai log -1 --format="โค้ดรุ่นล่าสุด: %h %s (%ad)" --date=short

# เช็กว่าโค้ดมีของที่รอบนี้ต้องใช้ (ถ้าไม่มี แปลว่ายังไม่ได้ push โค้ดล่าสุด)
train_code = open(CODE + "/train_yolo.py", encoding="utf-8").read()
for need in ["10_new_data", "12_new_data_seed3", "--new-test", "--compare", "seed_summary"]:
    if need not in train_code:
        raise Exception("train_yolo.py ใน GitHub ยังเป็นรุ่นเก่า (ไม่มี " + need + ") ให้ push โค้ดล่าสุดก่อน")
print("โค้ดครบ พร้อมเทรน")

## 4) ต่อ Google Drive แล้วโหลดข้อมูล
ผลเก็บใน Drive โฟลเดอร์ `KhadSan_detect_v3` (ไม่ทับรอบ 1-2)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/KhadSan_detect_v3"
os.makedirs(OUT, exist_ok=True)
print("ผลจะเก็บที่", OUT)

# ----- dataset หลัก (Fruits Quality) : ใช้ zip ใน Drive ถ้ามี ไม่มีก็โหลดจาก Roboflow ด้วย API key -----
SRC = "/content/FruitsQuality"
if os.path.exists(OUT + "/FruitsQuality.zip"):
    print("ใช้ FruitsQuality.zip จาก Drive")
    !unzip -q -o "{OUT}/FruitsQuality.zip" -d /content/zip_tmp
    for root, dirs, files in os.walk("/content/zip_tmp"):
        if "data.yaml" in files:
            SRC = root
            break
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("alex-day-qzfwa").project("fruits-quality-qtdtf")
    project.version(1).download("yolov8", location=SRC)

if not os.path.exists(SRC + "/data.yaml"):
    raise Exception("ไม่เจอ data.yaml ใน dataset หลัก เช็กไฟล์ zip หรือ API key")
count = 0
for part in ["train", "valid", "test"]:
    if os.path.isdir(SRC + "/" + part + "/images"):
        count = count + len(os.listdir(SRC + "/" + part + "/images"))
print("dataset หลัก", count, "รูป (ควรได้ 2583)")
if count != 2583:
    print("!!! จำนวนรูปไม่ตรงกับที่ทีมใช้ ผลอาจต่างจากเดิม")


# แตก zip แล้วหาโฟลเดอร์ที่มีชื่อตามที่ต้องการ
def unzip_find(zip_name, folder_name):
    if not os.path.exists(OUT + "/" + zip_name):
        raise Exception("รอบ 3 ต้องอัปโหลด " + zip_name + " ไว้ใน " + OUT + " ก่อน")
    tmp = "/content/tmp_" + zip_name.replace(".zip", "")
    !unzip -q -o "{OUT}/{zip_name}" -d "{tmp}"
    for root, dirs, files in os.walk(tmp):
        if os.path.basename(root) == folder_name:
            return root
    raise Exception("ใน " + zip_name + " ไม่มีโฟลเดอร์ " + folder_name)


def check_pairs(folder, should_be):
    images = len(os.listdir(folder + "/images"))
    labels = len(os.listdir(folder + "/labels"))
    print(os.path.basename(folder), images, "รูป (ควรได้", str(should_be) + ")")
    if images != labels:
        raise Exception("รูปกับไฟล์ป้ายไม่เท่ากันใน " + folder)


# ----- ชุดทดสอบภายนอก (Kaggle 600 รูป ไม่มีกรอบ) -----
EXTERNAL = unzip_find("ExternalTest.zip", "external_test")
total = 0
for c in sorted(os.listdir(EXTERNAL)):
    total = total + len(os.listdir(EXTERNAL + "/" + c))
print("external_test", total, "รูป (ควรได้ 600)")

# ----- รูปเทรนเพิ่ม (Kaggle + พื้นหลัง + ส้มเน่าใหม่) -----
EXTRA = unzip_find("KaggleBoxes_v3.zip", "kaggle_boxes_v3")
check_pairs(EXTRA, 3300)

# ----- ข้อสอบใหม่ 2 ชุด -----
NEW_TEST = unzip_find("NewTest.zip", "new_test")
SAME_SOURCE = unzip_find("NewTest.zip", "same_source_test")
check_pairs(NEW_TEST, 150)
check_pairs(SAME_SOURCE, 370)

## 5) เตรียมข้อมูล
แบ่งชุดเหมือนรอบ 1-2 (seed 42 ได้ชุดเดิมทุกครั้ง) แล้วทำ `data_more.yaml` = ชุดเทรนเดิม + รูปเทรนเพิ่ม (val และ test ใช้ชุดเดิม)

In [ ]:
!python "{CODE}/prepare_data.py" --src "{SRC}" --out /content/data_5fruits --extra "{EXTRA}"

for f in ["data.yaml", "data_more.yaml"]:
    if not os.path.exists("/content/data_5fruits/" + f):
        raise Exception("ไม่มี " + f + " เตรียมข้อมูลไม่สำเร็จ ดูข้อความ error ด้านบน")
for part in ["train", "val", "test"]:
    print(part, len(os.listdir("/content/data_5fruits/" + part + "/images")), "รูป (ควรได้ 1624 / 348 / 348)")
!cp /content/data_5fruits/dataset_counts.csv "{OUT}/"

## 6) เริ่มเทรน 3 การทดลอง แล้วสอบทุกชุด
ลำดับ : `10_new_data` (seed 42) → `11_new_data_seed2` → `12_new_data_seed3`
จากนั้นให้โมเดลที่เพิ่งเทรน + โมเดลเก่า 5, 8 สอบ test เดิม, ชุดแหล่งอื่น (Kaggle) และข้อสอบใหม่ 2 ชุด

In [ ]:
# โมเดลเก่าไว้เทียบ : 8 = ตัวที่เว็บใช้อยู่ (อยู่ใน GitHub), 5 = จาก Drive รอบ 2 (ไม่มีก็ข้าม)
OLD = {"8_cosine_more_data": "/content/khadsan-ai/web/models/best_detect.pt",
       "5_cosine_lr": "/content/drive/MyDrive/KhadSan_detect_v2/runs/5_cosine_lr/weights/best.pt"}
compare = []
for name in OLD:
    if os.path.exists(OLD[name]):
        compare.append(name + "=" + OLD[name])
    else:
        print("ไม่เจอโมเดล", name, "ข้ามการเทียบตัวนี้")
COMPARE = ",".join(compare)

!python "{CODE}/train_yolo.py" --data /content/data_5fruits/data.yaml --more-data /content/data_5fruits/data_more.yaml --only 10_new_data,11_new_data_seed2,12_new_data_seed3 --out "{OUT}" --external "{EXTERNAL}" --new-test "{NEW_TEST}" --same-source-test "{SAME_SOURCE}" --compare "{COMPARE}"

## 7) ดูผล

In [ ]:
import glob
import csv
from IPython.display import Image, Markdown, display

print("=== test เดิม (348 รูป) ===")
display(Markdown(open(OUT + "/results/summary.md", encoding="utf-8").read()))
print(open(OUT + "/models/class_info.json", encoding="utf-8").read())

print("=== 3 seed ผลแกว่งแค่ไหน ===")
if os.path.exists(OUT + "/results/seed_summary.md"):
    display(Markdown(open(OUT + "/results/seed_summary.md", encoding="utf-8").read()))

print("=== ข้อสอบใหม่ (rotten_called_fresh = ลูกเน่าที่โมเดลบอกว่าสด ยิ่งน้อยยิ่งดี) ===")
print(open(OUT + "/results/new_tests.csv", encoding="utf-8").read())

print("=== ชุดแหล่งอื่น (Kaggle) ===")
for f in sorted(glob.glob(OUT + "/results/external_*_summary.txt")):
    print(open(f, encoding="utf-8").read())

for name in ["compare_seeds", "confusion_matrix_normalized_best"]:
    file = OUT + "/results/" + name + ".png"
    if os.path.exists(file):
        print(name)
        display(Image(file))
print(open(OUT + "/results/per_class_best.csv", encoding="utf-8").read())

## 8) เช็กว่ายังตีกรอบหน้าคนเป็นผลไม้ไหม
ใช้รูปคน (zidane.jpg) กับรูปรถบัส (bus.jpg) ที่มากับ ultralytics ไม่มีผลไม้ในรูป ค่าที่ดีคือไม่เจอกรอบเลย
(ความมั่นใจขั้นต่ำ 0.40 เท่ากับที่เว็บใช้)

In [ ]:
from ultralytics.utils import ASSETS

for name in ["10_new_data", "11_new_data_seed2", "12_new_data_seed3"]:
    best = OUT + "/runs/" + name + "/weights/best.pt"
    if not os.path.exists(best):
        continue
    model = ultralytics.YOLO(best)
    for picture in ["zidane.jpg", "bus.jpg"]:
        r = model.predict(str(ASSETS / picture), conf=0.40, verbose=False)[0]
        found = []
        for i in range(len(r.boxes)):
            found.append(r.names[int(r.boxes.cls[i])] + " " + str(round(float(r.boxes.conf[i]) * 100)) + "%")
        if len(found) == 0:
            print(name, "|", picture, "-> ไม่เจอผลไม้ (ถูกต้อง)")
        else:
            print(name, "|", picture, "-> ตีกรอบผิด:", found)

## 9) เอาไปใช้ต่อ
- ส่งลิงก์โน้ตบุ๊กนี้ (ที่รันเสร็จแล้ว) ให้ทีมตรวจก่อน **ยังไม่ต้องเปลี่ยนโมเดลในเว็บ**
- ใช้ตัวใหม่แทนโมเดล 8 เมื่อครบ 4 ข้อ (เทียบกับแถว `8_cosine_more_data` ในตารางข้างบน)
  1. Test F1 (test เดิม) ไม่ตกมาก
  2. ชุดแหล่งอื่น (Kaggle) และ `new_test` ไม่แย่ลง
  3. ส้มเน่า (`per_class_best.csv`) ดีขึ้น และไม่มีผลไม้ไหนตกหนัก
  4. `rotten_called_fresh` ไม่เพิ่ม
- ถ้าใช้ตัวใหม่ : `models/best_detect.pt` + `class_info.json` → วางทับใน `web/models/`
- `KhadSan_detect_v3/results/` → วางใน `results/round3/` ของโปรเจกต์ (ไม่ต้องโหลด `runs/`)